In [ ]:
from flask import Flask, render_template, jsonify, request
import math

app = Flask(__name__)

# formation spacing (pixels between units)
SPACING = 120 

def create_object(id):
    # Calculate a unique station for each object in a row
    # Centers the row by offsetting based on the total count
    station_x = (id * SPACING) - ((4 - 1) * SPACING / 2)
    return {
        "id": id,
        "x": station_x, "y": -150.0,    # Start them "above" their stations
        "target_x": station_x,          # Each has a unique horizontal target
        "target_y": 0.0,
        "vx": 0.0, "vy": 0.0,
        "ix": 0.0, "iy": 0.0,
        "lx": 0.0, "ly": 0.0
    }

objects = [create_object(i) for i in range(4)]
state = {"mass": 1.2, "damping": 0.98}

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/update', methods=['POST'])
def update():
    data = request.json
    kp = float(data.get('kp', 10.7)) / 10.0
    kd = float(data.get('kd', 7.3)) / 10.0
    ki = float(data.get('ki', 3.0)) / 100.0
    disturbances = data.get('disturbances', [])

    output = []
    for i, obj in enumerate(objects):
        d = disturbances[i]
        
        # Error relative to their specific Station
        ex = obj["target_x"] - obj["x"]
        ey = obj["target_y"] - obj["y"]
        
        obj["ix"] += ex
        obj["iy"] += ey
        dx = ex - obj["lx"]
        dy = ey - obj["ly"]
        
        # Individual Control Force
        fcx = (ex * kp) + (obj["ix"] * ki) + (dx * kd)
        fcy = (ey * ey * ki if ey > 0 else ey * kp) # Simplified for vertical pull
        # Re-applying standard PID for simplicity:
        fcy = (ey * kp) + (obj["iy"] * ki) + (dy * kd)
        
        obj["vx"] = (obj["vx"] + (fcx + d['x']) / state["mass"]) * state["damping"]
        obj["vy"] = (obj["vy"] + (fcy + d['y']) / state["mass"]) * state["damping"]
        
        obj["x"] += obj["vx"]
        obj["y"] += obj["vy"]
        obj["lx"], obj["ly"] = ex, ey
        
        output.append({"id": i, "x": obj["x"], "y": obj["y"]})

    return jsonify(output)





if __name__ == '__main__':
    app.run(debug=True, port=8011, use_reloader = False)

In [ ]:
from flask import Flask, render_template, jsonify, request
import math

app = Flask(__name__)

# Constants for the LTI System
SPACING = 120  # Horizontal displacement between null equilibrium points
STATE = {
    "mass": 1.2,
    "damping": 0.98  # Viscous friction coefficient
}

def create_test_mass(id):
    """Initializes state vectors for a discrete test mass."""
    station_x = (id * SPACING) - ((4 - 1) * SPACING / 2)
    return {
        "id": id,
        "x": station_x, 
        "y": -150.0,       # Initial vertical perturbation
        "target_x": station_x,
        "target_y": 0.0,
        "vx": 0.0, 
        "vy": 0.0,
        "ix": 0.0,         # Error integral (X-axis)
        "iy": 0.0,         # Error integral (Y-axis)
        "lx": 0.0,         # Previous error state for derivative calculation
        "ly": 0.0
    }
    

# Fleet initialization (M0 through M3)
test_masses = [create_test_mass(i) for i in range(4)]

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/update', methods=['POST'])
def update():
    data = request.json
    
    # Extract coefficients alpha, beta, gamma (P, D, I)
    kp = float(data.get('kp', 10.7)) / 10.0
    kd = float(data.get('kd', 7.3)) / 10.0
    ki = float(data.get('ki', 3.0)) / 100.0
    
    perturbations = data.get('disturbances', [])

    output = []
    for i, m in enumerate(test_masses):
        f_p = perturbations[i] # External Perturbation Force
        
        # Calculate Error Signal e(t)
        ex = m["target_x"] - m["x"]
        ey = m["target_y"] - m["y"]
        
        # Accumulate Integral gamma
        m["ix"] += ex
        m["iy"] += ey
        
        # Calculate Derivative beta
        dx = ex - m["lx"]
        dy = ey - m["ly"]
        
        # Compute Compensatory Feedback Fc(t)
        fcx = (ex * kp) + (m["ix"] * ki) + (dx * kd)
        fcy = (ey * kp) + (m["iy"] * ki) + (dy * kd)
        
        # Integrate System Dynamics
        # a = (Fc + Fp) / M
        m["vx"] = (m["vx"] + (fcx + f_p['x']) / STATE["mass"]) * STATE["damping"]
        m["vy"] = (m["vy"] + (fcy + f_p['y']) / STATE["mass"]) * STATE["damping"]
        
        m["x"] += m["vx"]
        m["y"] += m["vy"]
        
        # Store state for next iteration
        m["lx"], m["ly"] = ex, ey
        
        # Calculate Euclidean Error Magnitude for Figure 1.0
        error_mag = math.sqrt(ex**2 + ey**2)

        
        
        output.append({
            "id": i,
            "x": m["x"],
            "y": m["y"],
            "error_mag": error_mag
        })

    return jsonify(output)

if __name__ == '__main__':
    # use_reloader=False prevents double-initialization in notebook environments
    app.run(debug=True, port=8011, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8011
Press CTRL+C to quit
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 15:50:44] "POST /update HTTP/1.1" 200 -
127.0